In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==========================================================
# NOTEBOOK 4
# Part A : Optimized Weighted Average (OWA)

import os
import warnings
import numpy as np
import pandas as pd

from scipy.optimize import minimize

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

warnings.filterwarnings("ignore")


# ==========================================================
# DIRECTORIES
# ==========================================================

PROJECT_DIR = "/content/drive/MyDrive/Biomass_Project"

DATA_DIR = os.path.join(PROJECT_DIR, "01_Data")

TABLE_DIR = os.path.join(PROJECT_DIR, "10_Tables", "Nested_CV")

OUTPUT_DIR = os.path.join(TABLE_DIR, "Advanced_Ensemble")

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ==========================================================
# BASE MODELS
# ==========================================================

MODELS = [
    "CatBoost",
    "RF",
    "XGB",
    "LGBM",
    "SVR"
]

print("Output Directory")
print(OUTPUT_DIR)

Output Directory
/content/drive/MyDrive/Biomass_Project/10_Tables/Nested_CV/Advanced_Ensemble


In [ ]:
# ==========================================================
# LOAD ORIGINAL DATASET
# ==========================================================

dataset_path = os.path.join(
    DATA_DIR,
    "Bangladesh_AGBD_RH95_Predictors_Final_Clean.csv"
)

data = pd.read_csv(dataset_path)

print(data.shape)

data.head()

(1227, 17)


,system:index,EVI,Elevation,NBR,NDMI,NDVI,Slope,VH,VV,agbd,rh95,VV_minus_VH,canopy_relief,Canopy_Cover,FHD,forest_name,PAI
0,0_0,0.446794,67,0.521174,0.262418,0.712379,5,-13.651795,-7.526714,133.741379,20.219999,6.125081,0.465772,0.811137,3.103661,Lawachara,3.337479
1,1_0,0.471140,72,0.589634,0.302198,0.784394,9,-13.769139,-5.908260,82.173004,25.500000,7.860879,0.695833,0.513298,2.886727,Lawachara,1.440718
2,2_0,0.521434,75,0.603416,0.335680,0.779537,3,-15.444711,-7.497519,82.173004,14.820000,7.947192,0.695833,0.513298,2.886727,Lawachara,1.440718
3,3_0,0.449244,71,0.593663,0.314374,0.757451,6,-13.838805,-7.893732,189.087280,32.540001,5.945073,0.541433,0.803823,3.229514,Lawachara,3.261589
4,4_0,0.359919,74,0.591196,0.313946,0.740298,1,-14.722586,-6.831360,79.405884,24.000000,7.891227,0.402498,0.825644,2.649440,Lawachara,3.497878


In [ ]:
# ==========================================================
# REPRESENTATIVE FEATURE LISTS
# ==========================================================

# ----------------------------------------------------------
# GEDI Structural Features
# ----------------------------------------------------------

gedi_features = [

    'rh95',
    'Canopy_Cover',
    'PAI',

]


# ----------------------------------------------------------
# Vegetation Indices
# ----------------------------------------------------------

veg_features = [

    'NDVI',
    'EVI',
    'NDMI',
    'NBR',

]


# ----------------------------------------------------------
# Sentinel-1 Backscatter
# ----------------------------------------------------------

sar_features = [

    'VV',
    'VH',
    'VV_minus_VH',

]


# ----------------------------------------------------------
# Terrain Features
# ----------------------------------------------------------

terrain_features = [

    'Elevation',
    'Slope',

]


# ==========================================================
# FINAL FEATURE LIST
# ==========================================================

FEATURE_COLUMNS = (
    gedi_features +
    veg_features +
    sar_features +
    terrain_features
)

TARGET = "agbd"

X_original = data[FEATURE_COLUMNS].copy()

y_original = data[TARGET].copy()

print("Number of predictors :", len(FEATURE_COLUMNS))
print("Feature matrix shape :", X_original.shape)
print("Target vector shape  :", y_original.shape)

Number of predictors : 12
Feature matrix shape : (1227, 12)
Target vector shape  : (1227,)


In [ ]:
# ==========================================================
# LOAD PREDICTION FILES
# ==========================================================

prediction_data = {}

for model in MODELS:

    path = os.path.join(
        TABLE_DIR,
        f"{model}_Predictions.csv"
    )

    prediction_data[model] = pd.read_csv(path)

reference = prediction_data["CatBoost"]

print("Prediction files loaded.")

Prediction files loaded.


In [ ]:
# ==========================================================
# VERIFY ALIGNMENT
# ==========================================================

for model in MODELS[1:]:

    df = prediction_data[model]

    assert reference["Index"].equals(df["Index"])
    assert reference["Observed"].equals(df["Observed"])
    assert reference["Fold"].equals(df["Fold"])

print("✓ Prediction alignment verified.")

✓ Prediction alignment verified.


In [ ]:
# ==========================================================
# ALIGN ORIGINAL DATASET
# ==========================================================

assert len(X_original) == len(reference), \
    "Dataset and prediction files have different lengths."

print("Original Features :", X_original.shape)
print("Prediction Samples:", reference.shape[0])

y_true = reference["Observed"].values

Original Features : (1227, 12)
Prediction Samples: 1227


In [ ]:
# ==========================================================
# OWA OPTIMIZATION FUNCTION
# ==========================================================

def optimize_weights(y_true, pred_matrix):
    """
    Optimize ensemble weights using constrained optimization.

    Parameters
    ----------
    y_true : ndarray
        Observed values.

    pred_matrix : ndarray
        Shape = (n_samples, n_models)

    Returns
    -------
    ndarray
        Optimal weights.
    """

    n_models = pred_matrix.shape[1]

    # ------------------------------------------------------
    # Objective Function
    # ------------------------------------------------------

    def objective(weights):

        prediction = pred_matrix @ weights

        rmse = np.sqrt(
            mean_squared_error(
                y_true,
                prediction
            )
        )

        return rmse

    # ------------------------------------------------------
    # Initial Weights
    # ------------------------------------------------------

    initial_weights = np.ones(n_models) / n_models

    # ------------------------------------------------------
    # Constraints
    # ------------------------------------------------------

    constraints = (
        {
            "type": "eq",
            "fun": lambda w: np.sum(w) - 1
        },
    )

    bounds = [(0, 1)] * n_models

    # ------------------------------------------------------
    # Optimization
    # ------------------------------------------------------

    result = minimize(
        objective,
        initial_weights,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints
    )

    if not result.success:
        raise RuntimeError(
            f"Optimization failed: {result.message}"
        )

    return result.x

In [ ]:
# ==========================================================
# MODEL EVALUATION FUNCTION
# ==========================================================

def evaluate_model(y_true, y_pred):

    r2 = r2_score(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    mape = np.mean(
        np.abs(
            (y_true - y_pred) / y_true
        )
    ) * 100

    rrmse = (
        rmse / np.mean(y_true)
    ) * 100

    return {
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "rRMSE": rrmse
    }


In [ ]:
# ==========================================================
# CREATE PREDICTION MATRIX
# ==========================================================

pred_matrix = np.column_stack([

    prediction_data[model]["Predicted"].values

    for model in MODELS

])

y_true = reference["Observed"].values

print("Prediction Matrix Shape :", pred_matrix.shape)

Prediction Matrix Shape : (1227, 5)


In [ ]:
# ==========================================================
# LEAKAGE-FREE OPTIMIZED WEIGHTED AVERAGE (OWA)
# ==========================================================

owa_pred = np.zeros(len(y_true))

weight_history = []

unique_folds = sorted(reference["Fold"].unique())

for fold in unique_folds:

    print(f"Optimizing Fold {fold}...")

    # ------------------------------------------------------
    # Training rows
    # ------------------------------------------------------

    train_mask = reference["Fold"] != fold

    X_train = pred_matrix[train_mask]

    y_train = y_true[train_mask]

    # ------------------------------------------------------
    # Test rows
    # ------------------------------------------------------

    test_mask = reference["Fold"] == fold

    X_test = pred_matrix[test_mask]

    # ------------------------------------------------------
    # Optimize weights
    # ------------------------------------------------------

    weights = optimize_weights(
        y_train,
        X_train
    )

    # ------------------------------------------------------
    # Save weights
    # ------------------------------------------------------

    weight_history.append({

        "Fold": fold,

        **{

            model: weight

            for model, weight

            in zip(MODELS, weights)

        }

    })

    # ------------------------------------------------------
    # Predict held-out fold
    # ------------------------------------------------------

    owa_pred[test_mask] = X_test @ weights

print("\nLeakage-free OWA completed.")

Optimizing Fold 1...
Optimizing Fold 2...
Optimizing Fold 3...
Optimizing Fold 4...
Optimizing Fold 5...

Leakage-free OWA completed.


In [ ]:
# ==========================================================
# EVALUATE OWA
# ==========================================================

owa_results = evaluate_model(
    y_true,
    owa_pred
)

print("\nOWA Performance")
print("-"*40)

for metric, value in owa_results.items():

    print(f"{metric:<8}: {value:.4f}")


OWA Performance
----------------------------------------
R2      : 0.7369
RMSE    : 32.2437
MAE     : 19.5893
MAPE    : 18.5800
rRMSE   : 27.5124


In [ ]:
# ==========================================================
# SAVE OWA WEIGHTS
# ==========================================================

weights_df = pd.DataFrame(weight_history)

weights_path = os.path.join(

    OUTPUT_DIR,

    "OWA_Fold_Weights.csv"

)

weights_df.to_csv(

    weights_path,

    index=False

)

print(weights_df.round(4))

   Fold  CatBoost      RF     XGB    LGBM     SVR
0     1    0.2985  0.2068  0.1624  0.1909  0.1414
1     2    0.3889  0.1246  0.2183  0.1871  0.0812
2     3    0.2582  0.1796  0.2914  0.1068  0.1641
3     4    0.2338  0.1988  0.2612  0.1788  0.1275
4     5    0.3071  0.2693  0.0000  0.1906  0.2330


In [ ]:
# ==========================================================
# SAVE OWA PREDICTIONS
# ==========================================================

owa_df = reference[

    ["Fold", "Index", "Observed"]

].copy()

owa_df["Predicted"] = owa_pred

owa_path = os.path.join(

    OUTPUT_DIR,

    "OWA_predictions.csv"

)

owa_df.to_csv(

    owa_path,

    index=False

)

print(f"Saved: {owa_path}")

Saved: /content/drive/MyDrive/Biomass_Project/10_Tables/Nested_CV/Advanced_Ensemble/OWA_predictions.csv


In [ ]:
# ==========================================================
# OWA WEIGHT SUMMARY
# ==========================================================

# Exclude the Fold column
weight_summary = (
    weights_df
    .drop(columns="Fold")
    .agg(["mean", "std", "min", "max"])
    .T
    .reset_index()
)

weight_summary.columns = [
    "Model",
    "Mean_Weight",
    "Std_Weight",
    "Min_Weight",
    "Max_Weight"
]

# Sort by average weight
weight_summary = weight_summary.sort_values(
    by="Mean_Weight",
    ascending=False
)

# Round values
weight_summary = weight_summary.round(4)

# Save
summary_path = os.path.join(
    OUTPUT_DIR,
    "OWA_Weight_Summary.csv"
)

weight_summary.to_csv(
    summary_path,
    index=False
)

print("=" * 60)
print("OWA Weight Summary")
print("=" * 60)
print(weight_summary)

print(f"\nSaved: {summary_path}")

OWA Weight Summary
      Model  Mean_Weight  Std_Weight  Min_Weight  Max_Weight
0  CatBoost       0.2973      0.0593      0.2338      0.3889
1        RF       0.1958      0.0521      0.1246      0.2693
2       XGB       0.1867      0.1151      0.0000      0.2914
3      LGBM       0.1708      0.0361      0.1068      0.1909
4       SVR       0.1494      0.0557      0.0812      0.2330

Saved: /content/drive/MyDrive/Biomass_Project/10_Tables/Nested_CV/Advanced_Ensemble/OWA_Weight_Summary.csv
